# Model charts

Render semantic composition and diagnostic charts without external plotting packages.

This sample uses only standard SysML v2 concepts and automatically discovers the project's `model/` or `src/` directory.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import syside

def find_sysml_root(start=Path.cwd()):
    """Find the nearest model/ or src/ folder containing textual SysML."""
    for directory in (start, *start.parents):
        for folder_name in ('model', 'src'):
            candidate = directory / folder_name
            if candidate.is_dir() and next(candidate.rglob('*.sysml'), None):
                return candidate
    raise FileNotFoundError('No model/ or src/ directory containing .sysml files was found')

SYSML_ROOT = find_sysml_root()
SYSML_FILES = sorted(SYSML_ROOT.rglob('*.sysml'))
model, diagnostics = syside.try_load_model([str(path) for path in SYSML_FILES])
print(f'Loaded {len(SYSML_FILES)} SysML files from {SYSML_ROOT}')

Loaded 42 SysML files from /Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model


In [2]:
from html import escape
from IPython.display import HTML, display

type_names = [
    'PartUsage', 'RequirementUsage', 'ActionUsage', 'PortUsage',
    'ConnectionUsage', 'StateUsage', 'AttributeUsage', 'ItemUsage',
    'PartDefinition', 'RequirementDefinition', 'ActionDefinition', 'PortDefinition'
]
counts = {}
for type_name in type_names:
    node_type = getattr(syside, type_name, None)
    if node_type is not None:
        counts[type_name] = len(list(model.elements(node_type)))

palette = ['#2d8d70', '#3887c7', '#e58a2b', '#8357b6', '#cf5c62', '#4f9d9d']
largest = max(counts.values(), default=1)
bars = []
for index, (name, count) in enumerate(sorted(counts.items(), key=lambda item: item[1], reverse=True)):
    width = max(1, count * 100 / largest)
    color = palette[index % len(palette)]
    bars.append(f'''<div style="display:grid;grid-template-columns:180px 1fr 70px;gap:12px;align-items:center;margin:7px 0">
      <div>{escape(name)}</div><div style="background:#edf2f4;border-radius:6px;overflow:hidden"><div style="width:{width:.1f}%;height:20px;background:{color}"></div></div><strong>{count:,}</strong>
    </div>''')

error_count = len(list(diagnostics.errors))
warning_count = len(list(diagnostics.warnings))
info_count = len(list(diagnostics.infos))
total_diagnostics = max(1, error_count + warning_count + info_count)
diagnostic_gradient = (
    f'#cf5c62 0 {error_count / total_diagnostics * 100:.1f}%, '
    f'#e5a52b {error_count / total_diagnostics * 100:.1f}% {(error_count + warning_count) / total_diagnostics * 100:.1f}%, '
    f'#3887c7 {(error_count + warning_count) / total_diagnostics * 100:.1f}% 100%'
)

display(HTML(f'''<div style="font-family:system-ui;max-width:900px">
  <h2>Semantic composition</h2>{''.join(bars)}
  <h2 style="margin-top:28px">Diagnostics</h2>
  <div style="display:flex;align-items:center;gap:24px">
    <div style="width:150px;height:150px;border-radius:50%;background:conic-gradient({diagnostic_gradient});position:relative">
      <div style="position:absolute;inset:28px;background:white;border-radius:50%;display:grid;place-items:center;font-size:24px;font-weight:700">{error_count + warning_count + info_count}</div>
    </div>
    <table><tr><th>Severity</th><th>Count</th></tr><tr><td>Errors</td><td>{error_count}</td></tr><tr><td>Warnings</td><td>{warning_count}</td></tr><tr><td>Information</td><td>{info_count}</td></tr></table>
  </div>
</div>'''))

Severity,Count
Errors,8009
Warnings,0
Information,0
